# Modern decoder-only transformer, English to German

Train this notebook on its own, after or before the Vaswani notebook. It does not train the other model.

The stack is Pre-RMSNorm, RoPE, grouped-query attention, QK-norm, and a SwiGLU mixture of experts.

Use a **T4 GPU** runtime. If this run shares the Drive folder with the Vaswani notebook, the data cell reuses the corpus. Curves land in the same Weights & Biases project, `transformer-scratch`, under the run name `modern_en_de`.

In [ ]:
import torch
print('torch', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime → Change runtime type → T4 GPU, then rerun.')
print('GPU', torch.cuda.get_device_name(0))

In [ ]:
import os
REPO = 'https://github.com/rupaktiwari/transformer-scratch.git'
if not os.path.exists('/content/transformer-scratch/mt'):
    !git clone {REPO} /content/transformer-scratch
%cd /content/transformer-scratch
!pip install -q uv
!uv pip install --system -r requirements-colab.txt

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/transformer-scratch'
DATA = ROOT + '/data'
SAVE = ROOT + '/checkpoints/modern_en_de'
os.makedirs(DATA, exist_ok=True)
os.makedirs(SAVE, exist_ok=True)

In [ ]:
import os
if not os.path.exists(DATA + '/meta.json'):
    !python -m mt.prepare_data --train-samples 200000 --out {DATA}
else:
    print('Reusing the corpus already on Drive')

In [ ]:
import wandb
wandb.login()

In [ ]:
!python -m mt.train --config configs/modern_en_de.yaml --data-dir {DATA} --save-dir {SAVE}

In [ ]:
!uv pip install --system 'unbabel-comet>=2.2.0'
!python -m mt.evaluate --config configs/modern_en_de.yaml --data-dir {DATA} --checkpoint {SAVE}/best.pt